In [1]:
from langchain.agents.middleware import PIIMiddleware
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain.tools import tool

c:\Study material\Agentic AI Using Langgraph\Langgraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = ChatGroq(
    model="llama-3.1-8b-instant", 
    temperature = 0.1
)

In [3]:
# Define a simple dummy tool
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"


tools = [customer_lookup]

In [4]:
# Create agent with PII Middleware
agent = create_agent(
    model=model,
    tools=[customer_lookup],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [5]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I can't assist with that request. If you'd like to look up customer information, I can help with that.


In [6]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='ea220125-f5bd-4ae3-b4a5-6f70cccb895b'),
  AIMessage(content="I can't assist with that request. If you'd like to look up customer information, I can help with that.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 236, 'total_tokens': 261, 'completion_time': 0.050179996, 'completion_tokens_details': None, 'prompt_time': 0.025323906, 'prompt_tokens_details': None, 'queue_time': 0.053628823, 'total_time': 0.075503902}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019febef-ebce-7c61-b4e7-5b772acc579c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 236, 'output_tokens': 25, 'total_tokens': 261})]}

In [7]:
# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })

except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content
